# Lesson 01: How does a machine learn a straight line?

**By the end you'll be able to**
- say what "training a model" actually means, without hand-waving
- derive gradient descent for linear regression and implement it in NumPy
- tell from a loss curve whether a learning rate is too small, too big or about right
- check gradient code numerically, the first debugging tool ML engineers reach for

**You need:** Python, a little NumPy, and first-year calculus (derivatives and the chain rule). No machine learning background.

**How to work through it**
- Read a section, then do the **Your turn** cell under it.
- 🔮 **Predict** means: write your guess in a comment *before* you run the cell. Being wrong is the useful part.
- Your implementation goes in `linreg.py`. Check it with `uv run pytest lessons/01-linear-regression`.
- Don't delete failed attempts. Comment them out, and note what went wrong in this lesson's README.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2
import linreg  # your code; autoreload picks up edits without restarting the kernel

## 1. The question

If the stock market goes up 1% today, how much does a particular stock tend to move?

Finance calls this number the stock's **beta** ($\beta$). A beta of 1.3 means the stock tends to move 1.3% for every 1% the market moves, so it amplifies the market. A beta of 0.5 means it's calmer than the market. Banks and funds use beta all the time, for example to measure how much market risk a portfolio carries, or how much to hedge.

Nobody hands you beta. You have to *estimate* it from data: lots of days, each with the market's return and the stock's return. That's what this lesson does, and the method you'll build is the same one, scaled up, that trains neural networks.

We'll start with **synthetic data**: fake data where we choose the true beta ourselves, so we can check whether the method finds it. With real data you never know the true answer, which is exactly why you test on fake data first.

In [ ]:
rng = np.random.default_rng(seed=0)

n_days = 250  # about one trading year
true_beta, true_alpha = 1.3, 0.02

market = rng.normal(loc=0.05, scale=1.0, size=n_days)  # the market's daily return, in %
noise = rng.normal(loc=0.0, scale=1.2, size=n_days)    # everything else that moves the stock
stock = true_alpha + true_beta * market + noise        # the stock's daily return, in %

market[:5], stock[:5]

**Your turn:** make a scatter plot with the market's return on the x-axis and the stock's return on the y-axis, with labelled axes. Would a straight line describe it? How spread out are the points around that line?

In [ ]:
# Your turn: scatter plot


## 2. A model is a guess with knobs

A **model** is a formula that turns an input into a prediction. The simplest one that could describe that scatter plot is a straight line:

$$\hat{y} = w x + b$$

- $x$ is the input: the market's return.
- $\hat{y}$ ("y-hat") is the model's *prediction* of the output, the stock's return. The hat means "estimated", to keep it separate from the real value $y$.
- $w$, the **weight**, is the slope. Here it plays the part of beta.
- $b$, the **bias** or intercept, shifts the line up or down. (Finance calls this one alpha.)

$w$ and $b$ are the model's **parameters**, its knobs. **Training** a model means finding the knob settings that make its predictions as good as possible on the data you have. That's the whole idea. The rest of this lesson is about doing it precisely.

**Your turn:** pick three $(w, b)$ pairs by eye and draw each line over your scatter plot. Which looks best? How would you *convince* someone it's the best, rather than just saying it looks right? That question is the next section.

In [ ]:
# Your turn: three hand-picked lines


## 3. Measuring "wrong": the loss function

For each day $i$, the **error** (or **residual**) is how far the prediction is from the truth: $\hat{y}_i - y_i$. We want one number that sums up how wrong the line is across all $n$ days.

*Attempt 1: add up the errors.* This fails. A prediction 2 too high and another 2 too low add up to 0, so a terrible line can get a perfect score.

*Attempt 2: add up the absolute errors* $|\hat{y}_i - y_i|$. This works, and it's sometimes used. But the absolute value has a sharp corner at 0, which makes the calculus in section 5 awkward.

*Attempt 3: square the errors and take the mean.* This is the **mean squared error** (MSE):

$$L(w, b) = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

Squaring makes every error count as positive, gives a smooth function that's easy to differentiate, and punishes big errors much more than small ones: an error of 3 costs 9, an error of 1 costs 1. That last property is a trade-off. A few extreme days, like a market crash, can drag the line around a lot. Keep that in mind.

A function that measures how wrong a model is, like this one, is called a **loss function** (or cost function). Notice that $L$ is written as a function of $w$ and $b$, not of $x$ and $y$. The data is fixed; the only things we can change are the knobs.

**Worked example.** Three days and the line $w = 1, b = 0$:

| $x$ | $y$ | $\hat{y} = 1 \cdot x + 0$ | error $\hat{y} - y$ | error² |
|---|---|---|---|---|
| 1 | 2 | 1 | −1 | 1 |
| 2 | 2 | 2 | 0 | 0 |
| 3 | 5 | 3 | −2 | 4 |

$L = (1 + 0 + 4) / 3 = 5/3 \approx 1.67$

> **A note on shapes.** `linreg.py` takes a 2D input `X` with shape `(n, d)`: `n` examples (days) by `d` features (inputs). We have one feature, so `X = market.reshape(-1, 1)` has shape `(250, 1)`, and `w` is an array with one entry. In matrix form the model is $\hat{y} = Xw + b$, which is just "$wx + b$ for every row at once" (`X @ w + b` in NumPy). It looks fussy for one input, but the same code will handle many inputs in section 9. Shape mistakes are the most common bug in ML code, so print `.shape` whenever you're unsure.

**Your turn**
1. Implement `predict` and `mse` in `linreg.py`, then run `uv run pytest lessons/01-linear-regression -k "Predict or MSE"`.
2. Compute the loss of your three hand-picked lines. Does the best-looking line have the lowest loss?

In [ ]:
X = market.reshape(-1, 1)
y = stock

# Your turn: loss of your three lines


## 4. The loss landscape

Keep the data fixed and think of $L$ as a function of the knobs. Every possible $(w, b)$ has a loss. Plot that and you get a landscape, and **training means finding its lowest point**.

For mean squared error with a straight-line model, the landscape is always a smooth bowl with a single bottom, because $L$ is a quadratic in $w$ and $b$. That's lucky: there's exactly one best answer and no false bottoms to get stuck in. Neural networks don't have this luxury, which is a story for lesson 09.

**Your turn**
1. Hold $b$ at 0. Compute the loss for 100 values of $w$ between −1 and 4, and plot loss against $w$. 🔮 Predict the shape, and roughly where the minimum is, before you run it.
2. Now vary both. Make a grid of $w$ and $b$ values with `np.meshgrid`, compute the loss at every point, and draw it with `plt.contour`. Where's the bottom of the bowl? Keep this plot; you'll draw on it in section 6.

In [ ]:
# 🔮 Prediction:

# Your turn: loss against w


In [ ]:
# Your turn: contour plot of the loss over (w, b)


## 5. Which way is downhill? The gradient

With two knobs you can search a grid, as you just did. A neural network has millions of knobs, where a grid is hopeless. So we need something smarter: from wherever we are, work out which direction is downhill and step that way.

"Which direction is downhill" is exactly what derivatives tell you. The **partial derivative** $\partial L / \partial w$ is the slope of the loss when you nudge $w$ and hold $b$ fixed. If it's positive, increasing $w$ increases the loss, so you want to decrease $w$. The partial derivatives together make the **gradient**:

$$\nabla L = \left( \frac{\partial L}{\partial w},\ \frac{\partial L}{\partial b} \right)$$

The gradient points uphill, in the direction of steepest increase, so downhill is *minus* the gradient.

**Try this on paper before reading on:** differentiate $L(w, b) = \frac{1}{n}\sum_i (w x_i + b - y_i)^2$ with respect to $w$, and then with respect to $b$. You only need the chain rule.

### Derivation (check yours against it)

Write $e_i = w x_i + b - y_i$ for the error on day $i$, so $L = \frac{1}{n}\sum_i e_i^2$.

By the chain rule, $\frac{\partial}{\partial w} e_i^2 = 2 e_i \frac{\partial e_i}{\partial w} = 2 e_i x_i$, and $\frac{\partial}{\partial b} e_i^2 = 2 e_i \cdot 1$. Averaging over the days:

$$\frac{\partial L}{\partial w} = \frac{2}{n}\sum_{i=1}^{n} (\hat{y}_i - y_i)\,x_i \qquad\qquad \frac{\partial L}{\partial b} = \frac{2}{n}\sum_{i=1}^{n} (\hat{y}_i - y_i)$$

In words: the gradient for $w$ is the average error, weighted by the input. If the line predicts too high ($\hat{y} > y$) on days when $x$ is large, $\partial L / \partial w$ is positive, so gradient descent will lower the slope. That matches intuition.

With $d$ features the same result in matrix form is $\nabla_w L = \frac{2}{n} X^\top (\hat{y} - y)$. Write it out for $d = 2$ to convince yourself it's the same thing.

**Your turn**
1. Implement `gradients` in `linreg.py`.
2. **Gradient check.** How do you know your derivative code is right? Compare it with a *numerical* estimate: nudge a parameter by a tiny $h$ in each direction and see how much the loss changes,
$$\frac{\partial L}{\partial w} \approx \frac{L(w + h) - L(w - h)}{2h}$$
Do it below for a few random $(w, b)$; the two answers should agree to about six decimal places. This is how ML engineers debug gradient code, and `TestGradients` does the same thing.
3. 🔮 What should the gradient be at the very bottom of the bowl? Check it.

In [ ]:
# Your turn: gradient check by hand


## 6. Gradient descent

Now the algorithm. Start from any guess, say $w = 0, b = 0$, and repeat:

$$w \leftarrow w - \alpha \frac{\partial L}{\partial w} \qquad\qquad b \leftarrow b - \alpha \frac{\partial L}{\partial b}$$

Each step moves the knobs a little way downhill. $\alpha$ is the **learning rate**: how big each step is. (Yes, finance uses $\alpha$ for something else. Context usually makes it clear.) Here every step uses all the data, so one step is one **epoch**, a full pass over the dataset.

A useful picture: you're on a hillside in thick fog, trying to reach the valley floor. You can't see the valley, but you can feel the slope under your feet. So you step in the steepest downhill direction, feel the slope again, and repeat. The learning rate is your stride length.

**Your turn**
1. Implement `fit` in `linreg.py`. It should record the loss at the start of every epoch, so you can watch training progress. Run `-k Fit`.
2. Fit the beta data with `lr=0.1, epochs=100`, and plot loss against epoch. This is a **loss curve**, and you'll look at a lot of them.
3. Record $(w, b)$ after every step (copy `fit` into the notebook, or give it an optional extra output) and draw the path on your contour plot from section 4. Does it head straight for the bottom?
4. How close did you get to the true beta of 1.3?

In [ ]:
# Your turn: fit, loss curve


In [ ]:
# Your turn: descent path on the contour plot


## 7. Breaking it on purpose: the learning rate

You'll run gradient descent for 50 epochs with five learning rates: `0.001`, `0.1`, `0.5`, `0.9` and `1.0`.

🔮 **Before running anything**, write down what you expect each loss curve to look like, and rank the five from fastest to slowest to converge.

Then run them all and plot the five loss curves on one chart. Use a log scale on the y-axis (`plt.yscale("log")`) so big and small losses both show up. Also draw the paths for `0.9` and `1.0` on your contour plot.

In [ ]:
# 🔮 Predictions:
# 0.001:
# 0.1:
# 0.5:
# 0.9:
# 1.0:


In [ ]:
# Your turn: five loss curves on one chart


### What's going on (read after you've run it)

- **Too small** (`0.001`): every step is safe but tiny. The loss barely moves in 50 epochs.
- **Good** (`0.1`, `0.5`): the loss drops quickly and smoothly.
- **Too big, but not yet unstable** (`0.9`): each step jumps *past* the bottom of the bowl and lands on the opposite slope. The next step jumps back. Each bounce is a bit smaller than the last, so it still gets there, but it's slower than `0.5`. Bigger isn't always faster.
- **Unstable** (`1.0`): each bounce lands *higher* up the other side than where it started. The loss grows every epoch, and given long enough it overflows to `inf` and then `nan`.

There's a sharp threshold between "bounces but converges" and "explodes". For this dataset it's about 0.94, which is why `0.9` and `1.0` behave so differently even though they're close. Where it sits depends on how steep the bowl is (stretch: find it by experiment, then see how it relates to the eigenvalues of $\frac{2}{n} \tilde{X}^\top \tilde{X}$, where $\tilde{X}$ is $X$ with a column of ones added).

Most people expect a bigger learning rate just to be faster. Write down what you predicted and what happened in the README's *What broke* section.

## 8. Checking against the exact answer

For linear regression specifically, you don't actually need gradient descent. Setting the gradient to zero and solving gives the answer directly. Add a column of ones to $X$ to handle $b$ (call the result $\tilde{X}$), and the solution is the **normal equation**:

$$\begin{pmatrix} w \\ b \end{pmatrix} = (\tilde{X}^\top \tilde{X})^{-1} \tilde{X}^\top y$$

So why learn gradient descent at all? Because almost no other model has a formula like this. Logistic regression (lesson 02) doesn't, and neural networks certainly don't. Gradient descent works for all of them. Linear regression is the one case where you can check gradient descent against an exact answer, which is why we start here.

**Your turn**
1. Implement `fit_closed_form`. Use `np.linalg.lstsq` or `np.linalg.solve` rather than computing the inverse with `np.linalg.inv`. Inverting is slower and less accurate, and on real data it can fail completely when two columns are nearly identical. Run `-k ClosedForm`.
2. Compare three numbers: the true beta (1.3), the closed-form estimate, and your gradient descent estimate. Gradient descent should match the closed form very closely.
3. 🔮 Why doesn't either of them come out at exactly 1.3? Test your explanation: what happens to the gap with `n_days = 25`, and with `n_days = 25_000`?

In [ ]:
# Your turn: closed form vs gradient descent vs the truth


## 9. More than one input, and a trap

Real models use many inputs, or **features**. Let's add the day's trading volume, measured in shares. The numbers are in the millions.

In [ ]:
volume = rng.normal(loc=5_000_000, scale=1_000_000, size=n_days)  # shares traded that day
X2 = np.column_stack([market, volume])
X2.shape

If you wrote `predict`, `gradients` and `fit` with matrix operations, they already handle two features.

**Your turn**
1. 🔮 Predict what happens with `linreg.fit(X2, stock, lr=0.1, epochs=100)`. Then run it.
2. Find a learning rate that doesn't blow up. How many epochs does it take for the market weight to get anywhere near the beta you found in section 8?
3. Now **standardise** each feature: subtract its mean and divide by its standard deviation, so every column has mean 0 and standard deviation 1. Fit again with `lr=0.1`. What changed, and why? Think back to the bowl in section 4. What shape is it when one knob is millions of times more sensitive than the other?
4. After standardising, the market weight is "per standard deviation of market return" rather than per 1%. How would you convert it back into a beta?

In [ ]:
# 🔮 Prediction:

# Your turn: unscaled features


In [ ]:
# Your turn: standardised features


### What's going on (read after you've tried it)

With unscaled features, the bowl turns into an extremely long, narrow valley. A tiny change to the volume weight changes the loss enormously, while the market weight hardly matters. A learning rate small enough not to blow up along the steep direction is far too small to make progress along the shallow one.

Standardising makes the bowl round again, so one learning rate suits every direction. Almost every real ML pipeline scales its features for exactly this reason.

## 10. Wrap-up

Fill in this lesson's README:
- **In my own words:** what does "training a model" mean? What are a loss function, a gradient and a learning rate? When would you use the closed form instead of gradient descent?
- **What broke:** the learning-rate and feature-scaling experiments, and anything else that went wrong.
- **Questions I still have.**

Then add the new terms to `GLOSSARY.md`: model, parameter, loss function, mean squared error, gradient, gradient descent, learning rate, epoch, feature, standardisation.

## Stretch (pick at least one)

- **A real beta.** Download about two years of daily closing prices for a stock and for an index (from [Stooq](https://stooq.com) or Yahoo Finance), turn them into daily % returns, and estimate the stock's beta. Compare it with the beta Yahoo Finance publishes. Why might they differ? Look up what period and frequency they use.
- **Derive the normal equation** by setting the matrix gradient to zero and solving.
- **Stochastic gradient descent.** Instead of using all 250 days for each step, use a random batch of 32. Plot the loss curve. Why is it noisy, and why does nearly all of deep learning do this anyway?
- **Outliers.** Add a crash day where the market falls 10% and the stock falls 40%. How far does the fitted beta move? Then try mean *absolute* error instead (you'll need its gradient) and compare.